# Step 3: Baseline - YOLOv8m
Unmodified YOLOv8m (COCO-pretrained). It is trained with the **same** hyper-parameters and explicit MRI augmentation as the improved model (`configs/train.yaml`), so any difference later is caused by the architecture only.

Set `QUICK_TEST = True` to check everything runs on a CPU (about 25% of the data, 3 epochs, 320 px). Use a GPU for the real run.

In [ ]:
# Set True for a fast CPU sanity run (3 epochs, 320 px, 25% of train). False = full experiment.
QUICK_TEST = False
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   # works from VS Code either way
sys.path.insert(0, str(ROOT / "src"))
import mri_detection as md
from mri_detection import data, viz, models, train, evaluate
cfg = md.load_config(quick=QUICK_TEST)
RUNS = md.runs_dir(cfg)
YOLO_DIR = md.ROOT / cfg["dataset"]["yolo_dir"]
DATA_YAML = data.training_yaml(cfg, YOLO_DIR)
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU) - use QUICK_TEST = True")
assert DATA_YAML.exists(), "Run 02_dataset_split.ipynb first"
import yaml; names = list(yaml.safe_load(open(DATA_YAML))["names"].values()); print(names)

## 3.1 Model size

In [ ]:
net = models.build_network("baseline", nc=len(names))
n_params = sum(p.numel() for p in net.parameters())
print(f"Baseline YOLOv8m: {n_params/1e6:.2f} M parameters")
net.info()

## 3.2 Training configuration (shared with the improved model)

In [ ]:
args = md.config.train_args(cfg, DATA_YAML, RUNS)
for k, v in args.items(): print(f"{k:16s}: {v}")

## 3.3 Train

In [ ]:
BEST_BASE = train.train_baseline(cfg, DATA_YAML, RUNS, name="baseline_yolov8m")
print("Saved:", BEST_BASE)

## 3.4 Quick look at the training curves

In [ ]:
import matplotlib.pyplot as plt
c = evaluate.load_curve(RUNS, "baseline_yolov8m")
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
for a, (col, t) in zip(ax, [("train/box_loss", "Train box loss"), ("metrics/mAP50(B)", "Val mAP@50"), ("metrics/mAP50-95(B)", "Val mAP@50-95")]):
    a.plot(c["epoch"], c[col]); a.set_title(t); a.set_xlabel("epoch"); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

Next: `04_improved_architecture.ipynb`.